In [1]:
!pip install evaluate[all] rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=bd0a720f41ca350649e7abd2c281590ebb2542084418c814b156ba61056197f4
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
# Для метрик потрібна бібліотека:
# pip install evaluate

import math, random
from typing import List, Tuple
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import evaluate

# ----------------------- Config -----------------------
SEED = 123
random.seed(SEED); torch.manual_seed(SEED)  # Фіксуємо генератори випадкових чисел для відтворюваності
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # Використання GPU, якщо доступне

# Маленький корпус речень, який буде розмножено для тренування
CORPUS = [
    "the quick brown fox jumps over the lazy dog",
    "language models can generate coherent text",
    "transformers are powerful for sequence modeling",
    "attention is all you need for many tasks",
    "pytorch includes transformer encoder and decoder layers",
    "we build a tiny decoder only transformer for text",
    "greedy decoding often stops at sentence boundary",
    "larger datasets improve generalization and diversity",
    "validation perplexity correlates with sample quality",
    "this minimal example runs fast on cpu",
] * 200  # множимо для більшого датасету

# Основні гіперпараметри
MAX_TOKENS = 32
BATCH_SIZE = 64
EPOCHS = 20
LR = 2e-3
D_MODEL = 128
NHEAD = 4
NUM_LAYERS = 2
FFN_DIM = 256

# -------------------- Tokenizer (split) --------------------
# Спеціальні токени
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD, BOS, EOS, UNK = 0, 1, 2, 3  # їх індекси

def normalize_text(s: str) -> str:
    # Приводимо рядок до нижнього регістру й видаляємо зайві пробіли
    return " ".join(s.lower().strip().split())

def build_vocab(texts: List[str], min_freq: int = 1):
    # Побудова словника (word2id і id2word)
    from collections import Counter
    cnt = Counter()
    for s in texts:
        for w in normalize_text(s).split():
            cnt[w] += 1
    itos = SPECIALS[:] + sorted([w for w, f in cnt.items() if f >= min_freq])
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

stoi, itos = build_vocab(CORPUS)
VOCAB_SIZE = len(itos)

def encode_tokens(text: str, max_len: int = MAX_TOKENS) -> List[int]:
    # Перетворення тексту на список індексів із BOS та EOS
    toks = [BOS] + [stoi.get(w, UNK) for w in normalize_text(text).split()] + [EOS]
    toks = toks[:max_len] + [PAD] * max(0, max_len - len(toks))  # паддінг
    return toks

def decode_tokens(ids: List[int]) -> str:
    # Відновлення тексту з індексів, пропускаючи службові токени
    words = []
    for i in ids:
        if i in (PAD, BOS, EOS):
            continue
        words.append(itos[i])
    return " ".join(words)

# ----------------------- Dataset -----------------------
class LanguageModelingDataset(Dataset):
    def __init__(self, texts: List[str], max_len: int = MAX_TOKENS):
        # Кожне речення кодується в токени фіксованої довжини
        self.samples: List[List[int]] = [encode_tokens(t, max_len) for t in texts]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx) -> Tuple[torch.Tensor, torch.Tensor]:
        # Вхід (x) — все окрім останнього токена
        # Ціль (y) — все окрім першого токена
        ids = torch.tensor(self.samples[idx], dtype=torch.long)  # (T)
        x = ids[:-1]                                             # (T-1)
        y = ids[1:]                                              # (T-1)
        return x, y

# Розбиття на train/val
random.shuffle(CORPUS)
split = int(0.9 * len(CORPUS))
train_ds = LanguageModelingDataset(CORPUS[:split], MAX_TOKENS)
val_ds   = LanguageModelingDataset(CORPUS[split:], MAX_TOKENS)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# ----------------------- Model -----------------------
class SinusoidalPE(nn.Module):
    # Класичне позиційне кодування
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, L, D)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1), :]

class DecoderOnlyLM(nn.Module):
    # Спрощена GPT-подібна LM
    def __init__(self, vocab_size: int, d_model=D_MODEL, nhead=NHEAD, layers=NUM_LAYERS, ffn=FFN_DIM):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pe  = SinusoidalPE(d_model, MAX_TOKENS)
        dec = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward=ffn, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec, num_layers=layers)
        self.proj = nn.Linear(d_model, vocab_size)  # проекція в простір словника

    @staticmethod
    def causal_mask(T: int, device) -> torch.Tensor:
        # Забороняє "зазирнути вперед" у майбутні токени
        return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

    def forward(self, x_in: torch.Tensor) -> torch.Tensor:
        h = self.pe(self.emb(x_in))           # (B,T,D)
        T = h.size(1)
        mask = self.causal_mask(T, h.device)  # (T,T)
        memory = torch.zeros((h.size(0), 1, h.size(2)), device=h.device)  # "фейкова" пам’ять
        out = self.decoder(h, memory, tgt_mask=mask)  # (B,T,D)
        return self.proj(out)                  # (B,T,V)

# Модель, оптимізатор і функція втрат
model = DecoderOnlyLM(VOCAB_SIZE).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

# ----------------------- Generation -----------------------
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 30) -> str:
    # Жадібна генерація токен за токеном
    model.eval()
    init = [BOS] + [stoi.get(w, UNK) for w in normalize_text(prompt).split()]
    init = init[:MAX_TOKENS - 2]
    x = torch.tensor(init, dtype=torch.long, device=DEVICE).unsqueeze(0)  # (1,T0)
    for _ in range(max_new_tokens):
        if x.size(1) >= MAX_TOKENS - 1:
            break
        logits = model(x)                   # (1,T,V)
        next_id = logits[0, -1].argmax()   # greedy
        x = torch.cat([x, next_id.view(1,1)], dim=1)
        if next_id.item() == EOS:
            break
    return decode_tokens(x[0].tolist())

# ----------------------- Metrics (evaluate) -----------------------
# Використовуємо бібліотеку evaluate від HuggingFace
bleu_metric  = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

def compute_bleu(pred_texts: List[str], ref_texts: List[str]) -> float:
    # Обчислення BLEU для згенерованих текстів
    if not pred_texts or not ref_texts or len(pred_texts) != len(ref_texts):
        return 0.0
    filtered_preds, filtered_refs = [], []
    for pred, ref in zip(pred_texts, ref_texts):
        if pred.strip() and ref.strip():
            filtered_preds.append(pred.strip())
            filtered_refs.append([ref.strip()])
    if not filtered_preds:
        return 0.0
    result = bleu_metric.compute(predictions=filtered_preds, references=filtered_refs)
    return float(result["bleu"])

def compute_rouge_l(pred_texts: List[str], ref_texts: List[str]) -> float:
    # Обчислення ROUGE-L
    if not pred_texts or not ref_texts:
        return 0.0
    preds = [str(p) for p in pred_texts]
    refs  = [str(r) for r in ref_texts]
    out = rouge_metric.compute(predictions=preds, references=refs)
    return float(out["rougeL"])

# ----------------------- Train / Eval -----------------------
def run_epoch(dataloader, train: bool):
    # Одина епоха — тренування або валідація
    model.train(train)
    total_loss, total_tokens = 0.0, 0
    for x, y in dataloader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if train:
            optimizer.zero_grad()
        logits = model(x)  # (B,T,V)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        if train:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
            optimizer.step()
        valid = (y != PAD)
        total_loss += (loss.item() * valid.sum().item())
        total_tokens += valid.sum().item()
    ce = total_loss / max(1, total_tokens)
    ppl = math.exp(ce)
    return ce, ppl

@torch.no_grad()
def evaluate_with_text_metrics(dataloader):
    # Валідація з підрахунком BLEU та ROUGE-L
    model.eval()
    total_loss, total_tokens = 0.0, 0
    pred_texts, ref_texts = [], []

    for x, y in dataloader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        total_loss += loss.item() * x.numel()
        total_tokens += x.numel()

        for i in range(x.size(0)):
            # prompt = усе до EOS або PAD
            xtok = x[i].tolist()
            if EOS in xtok:
                xtok = xtok[:xtok.index(EOS)]
            elif PAD in xtok:
                xtok = xtok[:xtok.index(PAD)]
            prompt_str = decode_tokens(xtok)

            # reference continuation = таргетні токени до EOS/PAD
            ytok = y[i].tolist()
            if EOS in ytok:
                ytok = ytok[:ytok.index(EOS)]
            elif PAD in ytok:
                ytok = ytok[:ytok.index(PAD)]
            reference_str = decode_tokens(ytok)
            if not reference_str.strip():
                continue

            # згенерований текст
            gen_str = generate(prompt_str, max_new_tokens=max(1, len(ytok)))

            # Відкидаємо prompt, залишаємо лише continuation
            gen_words = gen_str.split()
            prompt_words = prompt_str.split()
            if len(gen_words) > len(prompt_words):
                gen_continuation = ' '.join(gen_words[len(prompt_words):])
            else:
                gen_continuation = gen_str

            pred_texts.append(gen_continuation.strip())
            ref_texts.append(reference_str.strip())

    ce = total_loss / max(1, total_tokens)
    ppl = math.exp(ce)
    bleu = compute_bleu(pred_texts, ref_texts) if pred_texts and ref_texts else 0.0
    rouge_l = compute_rouge_l(pred_texts, ref_texts) if pred_texts and ref_texts else 0.0
    return ce, ppl, bleu, rouge_l

# ----------------------- Loop -----------------------
for epoch in range(1, EPOCHS + 1):
    tr_ce, tr_ppl = run_epoch(train_dl, train=True)
    va_ce, va_ppl, va_bleu, va_rouge = evaluate_with_text_metrics(val_dl)
    print(f"epoch {epoch:02d} | train CE/tok={tr_ce:.4f} PPL={tr_ppl:.2f} "
          f"| val CE/tok={va_ce:.4f} PPL={va_ppl:.2f} BLEU={va_bleu:.4f} ROUGE-L={va_rouge:.4f}")

print("\n--- sample ---")
print(generate("language can", max_new_tokens=20))

epoch 01 | train CE/tok=1.1534 PPL=3.17 | val CE/tok=0.3081 PPL=1.36 BLEU=1.0000 ROUGE-L=1.0000
epoch 02 | train CE/tok=0.3078 PPL=1.36 | val CE/tok=0.2966 PPL=1.35 BLEU=1.0000 ROUGE-L=1.0000
epoch 03 | train CE/tok=0.2975 PPL=1.35 | val CE/tok=0.2923 PPL=1.34 BLEU=1.0000 ROUGE-L=1.0000
epoch 04 | train CE/tok=0.2958 PPL=1.34 | val CE/tok=0.2849 PPL=1.33 BLEU=1.0000 ROUGE-L=1.0000
epoch 05 | train CE/tok=0.2936 PPL=1.34 | val CE/tok=0.2889 PPL=1.33 BLEU=1.0000 ROUGE-L=1.0000
epoch 06 | train CE/tok=0.2931 PPL=1.34 | val CE/tok=0.2840 PPL=1.33 BLEU=1.0000 ROUGE-L=1.0000
epoch 07 | train CE/tok=0.2922 PPL=1.34 | val CE/tok=0.2835 PPL=1.33 BLEU=1.0000 ROUGE-L=1.0000
epoch 08 | train CE/tok=0.2917 PPL=1.34 | val CE/tok=0.2862 PPL=1.33 BLEU=1.0000 ROUGE-L=1.0000
epoch 09 | train CE/tok=0.2902 PPL=1.34 | val CE/tok=0.2848 PPL=1.33 BLEU=1.0000 ROUGE-L=1.0000
epoch 10 | train CE/tok=0.2905 PPL=1.34 | val CE/tok=0.2866 PPL=1.33 BLEU=1.0000 ROUGE-L=1.0000
epoch 11 | train CE/tok=0.2901 PPL=1.34 